# Higher-Order Greeks and Cross-Greeks\n\n**Level:** Advanced\n**Concepts Covered:** 6\n\n---\n\n## Overview\n\nThis comprehensive notebook covers higher-order greeks and cross-greeks with detailed explanations, mathematical derivations, Python implementations, and practical examples.

## 1. Introduction to Higher-Order Greeks

### Why Higher-Order Greeks Matter

While first-order Greeks (Delta, Vega) measure linear sensitivities of option prices, **higher-order Greeks** capture non-linear behaviors that become critical in real-world risk management:

1. **P&L Attribution**: Explain daily portfolio changes beyond simple delta moves
2. **Risk Management**: Capture convexity risks ignored by first-order approximations
3. **Model Calibration**: Volatility smile dynamics require cross-Greeks (Vanna, Volga)
4. **Dynamic Hedging**: Understand how Greeks themselves change (Charm = ∂Δ/∂t)

### Greek Hierarchy

**First-Order Greeks** (Linear Sensitivities):
- **Delta (Δ)**: ∂C/∂S - Sensitivity to spot price
- **Vega (ν)**: ∂C/∂σ - Sensitivity to volatility
- **Theta (Θ)**: ∂C/∂t - Time decay
- **Rho (ρ)**: ∂C/∂r - Interest rate sensitivity

**Second-Order Greeks** (Curvature):
- **Gamma (Γ)**: ∂²C/∂S² = ∂Δ/∂S - Delta convexity
- **Volga/Vomma**: ∂²C/∂σ² = ∂ν/∂σ - Vega convexity

**Cross-Greeks** (Interaction Effects):
- **Vanna**: ∂²C/∂S∂σ = ∂Δ/∂σ = ∂ν/∂S - Spot-vol interaction
- **Charm**: ∂²C/∂S∂t = ∂Δ/∂t - Delta decay
- **Veta/DvegaDtime**: ∂²C/∂σ∂t = ∂ν/∂t - Vega decay

### Taylor Series Expansion for Option Value Changes

The option price change can be decomposed using Taylor series:

$$
\Delta C \approx \underbrace{\Delta \cdot \Delta S}_{\text{Delta}} + \underbrace{\frac{1}{2}\Gamma \cdot (\Delta S)^2}_{\text{Gamma}} + \underbrace{\nu \cdot \Delta \sigma}_{\text{Vega}} + \underbrace{\Theta \cdot \Delta t}_{\text{Theta}} + \underbrace{\rho \cdot \Delta r}_{\text{Rho}}
$$

$$
+ \underbrace{\text{Vanna} \cdot \Delta S \cdot \Delta \sigma}_{\text{Cross-term}} + \underbrace{\frac{1}{2}\text{Volga} \cdot (\Delta \sigma)^2}_{\text{Vol convexity}} + \underbrace{\text{Charm} \cdot \Delta S \cdot \Delta t}_{\text{Delta decay}} + \ldots
$$

**Real-World Applications:**

1. **Market Makers**: Gamma scalping profits from volatility, need accurate Gamma
2. **Exotic Desks**: Cross-Greeks essential for barrier, asian, lookback options
3. **Volatility Trading**: Vanna/Volga for skew trading, vega hedging
4. **Risk Aggregation**: Portfolio Greeks across thousands of positions

### Learning Objectives

By the end of this notebook, you will:
1. Derive and implement all second-order Greeks analytically
2. Understand cross-Greeks and their role in volatility smile dynamics
3. Build gamma scalping and theta collection simulators
4. Apply Vanna-Volga pricing methodology
5. Perform comprehensive P&L attribution using full Taylor expansion
6. Manage a complex options portfolio with dynamic hedging

Let's begin with **Gamma**, the most important second-order Greek.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
# Colorblind-friendly palette
colors = ['#0173B2', '#DE8F05', '#029E73', '#CC78BC', '#CA9161', '#949494']
sns.set_palette(colors)
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

## 2. Gamma: Convexity Risk

### Theory: The Most Important Second-Order Greek

**Gamma (Γ)** measures the rate of change of Delta with respect to the underlying price:

$$
\Gamma = \frac{\partial^2 C}{\partial S^2} = \frac{\partial \Delta}{\partial S}
$$

**Key Properties:**
1. **Always positive** for long calls and puts (convexity benefit)
2. **Highest at-the-money (ATM)** - peaks when S ≈ K
3. **Decreases as expiry approaches** - gamma risk explodes near maturity
4. **Same for calls and puts** (European options): Γ_call = Γ_put

**Why Gamma Matters:**

1. **Delta Hedging Frequency**: Higher gamma means delta changes faster → need more frequent rebalancing
2. **Gamma Scalping P&L**: Market makers profit from realized volatility through gamma
   - Long gamma: profit from large moves (scalping gains)
   - Short gamma: profit from range-bound markets (collect premium)
3. **P&L Convexity**: Gamma creates non-linear P&L in the underlying
4. **Risk Management**: Gamma explosions near expiry can cause large unexpected losses

**Gamma-Theta Relationship:**

For delta-neutral portfolios, Gamma and Theta have opposite signs (fundamental Black-Scholes PDE):

$$
\Theta + \frac{1}{2}\sigma^2 S^2 \Gamma + rS\Delta - rC = 0
$$

When delta-hedged (Δ = 0):
$$
\Theta \approx -\frac{1}{2}\sigma^2 S^2 \Gamma
$$

**Interpretation**: 
- Long gamma → negative theta (pay time decay for convexity)
- Short gamma → positive theta (collect time decay, risk convexity)

### Mathematical Derivation: Black-Scholes Gamma

Starting from Black-Scholes call delta:
$$
\Delta_{\text{call}} = N(d_1), \quad \text{where} \quad d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}
$$

Taking the derivative with respect to S:
$$
\Gamma = \frac{\partial \Delta}{\partial S} = \frac{\partial N(d_1)}{\partial S} = N'(d_1) \cdot \frac{\partial d_1}{\partial S}
$$

where $N'(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ is the standard normal PDF.

Computing $\frac{\partial d_1}{\partial S}$:
$$
\frac{\partial d_1}{\partial S} = \frac{1}{S\sigma\sqrt{T}}
$$

**Final Black-Scholes Gamma Formula:**
$$
\boxed{\Gamma = \frac{N'(d_1)}{S\sigma\sqrt{T}} = \frac{\phi(d_1)}{S\sigma\sqrt{T}}}
$$

where $\phi(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ is the standard normal density.

**Key Insights:**
1. Gamma is proportional to $N'(d_1)$ - the probability density of being ATM at expiry
2. Gamma is inversely proportional to $S\sigma\sqrt{T}$ - explodes as T → 0
3. **Call and put have identical gamma** (second derivative doesn't depend on K)
4. Maximum gamma when $d_1 = 0$ (ATM condition: $S = K e^{-(r+\sigma^2/2)T} \approx K$)

In [ ]:
# Python Implementation: Black-Scholes Greeks

def black_scholes_call(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculate Black-Scholes call option price.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity (years)
    r : float
        Risk-free rate (annualized)
    sigma : float
        Volatility (annualized)
    
    Returns
    -------
    float
        Call option price
    """
    if T <= 0:
        return max(S - K, 0)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price


def black_scholes_put(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculate Black-Scholes put option price.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity (years)
    r : float
        Risk-free rate (annualized)
    sigma : float
        Volatility (annualized)
    
    Returns
    -------
    float
        Put option price
    """
    if T <= 0:
        return max(K - S, 0)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price


def gamma_bs(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculate Black-Scholes Gamma (same for calls and puts).
    
    Gamma measures the rate of change of Delta with respect to the underlying price:
    Γ = ∂²C/∂S² = ∂Δ/∂S = φ(d₁)/(S·σ·√T)
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity (years)
    r : float
        Risk-free rate (annualized)
    sigma : float
        Volatility (annualized)
    
    Returns
    -------
    float
        Gamma value (same for calls and puts)
    
    Notes
    -----
    - Always positive for long options
    - Peaks at ATM (S ≈ K)
    - Explodes as T → 0
    - Measures convexity of option value
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    return gamma


def delta_bs(S: float, K: float, T: float, r: float, sigma: float, option_type: str = 'call') -> float:
    """
    Calculate Black-Scholes Delta.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity (years)
    r : float
        Risk-free rate (annualized)
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Delta value
    """
    if T <= 0:
        if option_type == 'call':
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    if option_type == 'call':
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1


def gamma_scalping_pnl(S_path: np.ndarray, gamma: float, delta_initial: float) -> Tuple[float, np.ndarray]:
    """
    Simulate gamma scalping P&L from delta hedging.
    
    The P&L from gamma scalping comes from rebalancing the delta hedge:
    P&L ≈ 0.5 * Γ * (ΔS)² - hedging costs
    
    Parameters
    ----------
    S_path : np.ndarray
        Simulated stock price path
    gamma : float
        Portfolio gamma
    delta_initial : float
        Initial delta position
    
    Returns
    -------
    tuple
        (total_pnl, pnl_series) - Total P&L and cumulative P&L over time
    
    Notes
    -----
    Gamma scalping exploits the convexity of options:
    - Profits from volatility (realized vol > implied vol)
    - Requires frequent rebalancing
    - Transaction costs reduce actual P&L
    """
    n_steps = len(S_path)
    pnl = np.zeros(n_steps)
    
    for i in range(1, n_steps):
        dS = S_path[i] - S_path[i-1]
        # Gamma P&L: 0.5 * Gamma * (dS)^2
        pnl[i] = pnl[i-1] + 0.5 * gamma * dS**2
    
    return pnl[-1], pnl


print('[OK] Black-Scholes pricing and Gamma functions implemented')

# Test
S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.25
call_price = black_scholes_call(S, K, T, r, sigma)
put_price = black_scholes_put(S, K, T, r, sigma)
gamma = gamma_bs(S, K, T, r, sigma)
delta_call = delta_bs(S, K, T, r, sigma, 'call')
delta_put = delta_bs(S, K, T, r, sigma, 'put')

print(f'\nTest (S={S}, K={K}, T={T}, r={r}, σ={sigma}):')
print(f'Call Price: ${call_price:.4f}')
print(f'Put Price:  ${put_price:.4f}')
print(f'Gamma:      {gamma:.6f} (same for call and put)')
print(f'Delta Call: {delta_call:.4f}')
print(f'Delta Put:  {delta_put:.4f}')

In [ ]:
# Visualization: Gamma Behavior (2x2 subplots)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Gamma: Convexity Risk Analysis', fontsize=16, fontweight='bold', y=1.00)

# Base parameters
S0 = 100
r = 0.05
sigma = 0.25

# Panel 1: Gamma across strikes (different maturities)
ax = axes[0, 0]
spot_range = np.linspace(60, 140, 200)
maturities = [0.08, 0.25, 0.5, 1.0]  # 1 month, 3 months, 6 months, 1 year

for T in maturities:
    gammas = [gamma_bs(S, 100, T, r, sigma) for S in spot_range]
    ax.plot(spot_range, gammas, linewidth=2, label=f'T = {T:.2f}y ({int(T*252)}d)')

ax.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='Strike K=100')
ax.set_xlabel('Spot Price ($)', fontsize=11)
ax.set_ylabel('Gamma', fontsize=11)
ax.set_title('Gamma vs Spot Price (ATM peaks, shorter T = higher Γ)', fontsize=12)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 2: Gamma vs time to expiry (different moneyness)
ax = axes[0, 1]
time_range = np.linspace(0.02, 2.0, 200)
strikes = [90, 95, 100, 105, 110]

for K in strikes:
    gammas = [gamma_bs(S0, K, T, r, sigma) for T in time_range]
    moneyness = K / S0
    ax.plot(time_range, gammas, linewidth=2, label=f'K={K} (M={moneyness:.2f})')

ax.set_xlabel('Time to Expiry (years)', fontsize=11)
ax.set_ylabel('Gamma', fontsize=11)
ax.set_title('Gamma vs Time (Gamma explosion as T→0 for ATM)', fontsize=12)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 3: Gamma scalping P&L simulation
ax = axes[1, 0]
T = 30/252  # 30 days
dt = 1/252  # daily rebalancing
n_steps = 30
time_grid = np.linspace(0, T, n_steps)

# Simulate 3 price paths with different realized volatilities
np.random.seed(42)
scenarios = [
    ('Low Vol (σ=15%)', 0.15, 'blue'),
    ('Med Vol (σ=25%)', 0.25, 'green'),
    ('High Vol (σ=35%)', 0.35, 'red')
]

for label, realized_vol, color in scenarios:
    # Simulate GBM price path
    Z = np.random.randn(n_steps)
    dS = S0 * realized_vol * np.sqrt(dt) * Z
    S_path = S0 + np.cumsum(dS)
    S_path = np.insert(S_path, 0, S0)
    
    # Calculate gamma scalping P&L
    gamma_val = gamma_bs(S0, 100, T, r, sigma)
    _, pnl_series = gamma_scalping_pnl(S_path, gamma_val, 0.5)
    
    ax.plot(range(len(pnl_series)), pnl_series, linewidth=2, label=label, color=color)

ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Trading Days', fontsize=11)
ax.set_ylabel('Cumulative P&L ($)', fontsize=11)
ax.set_title('Gamma Scalping P&L (Long ATM Straddle, 30 days)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.text(0.02, 0.98, 'Long gamma profits from volatility', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Panel 4: Gamma-Theta relationship (tradeoff)
ax = axes[1, 1]
spot_range = np.linspace(80, 120, 100)
T = 0.25

gammas = [gamma_bs(S, 100, T, r, sigma) for S in spot_range]
# Theta approximation: Θ ≈ -0.5 * σ² * S² * Γ (for delta-neutral)
thetas = [-0.5 * sigma**2 * S**2 * gamma_bs(S, 100, T, r, sigma) for S in spot_range]

ax2 = ax.twinx()
line1 = ax.plot(spot_range, gammas, linewidth=2.5, label='Gamma (left axis)', color=colors[0])
line2 = ax2.plot(spot_range, thetas, linewidth=2.5, label='Theta (right axis)', color=colors[1], linestyle='--')

ax.axvline(x=100, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Spot Price ($)', fontsize=11)
ax.set_ylabel('Gamma', fontsize=11, color=colors[0])
ax2.set_ylabel('Theta ($/day)', fontsize=11, color=colors[1])
ax.set_title('Gamma-Theta Tradeoff (Long gamma = negative theta)', fontsize=12)
ax.tick_params(axis='y', labelcolor=colors[0])
ax2.tick_params(axis='y', labelcolor=colors[1])
ax.grid(True, alpha=0.3)

# Combined legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper right')

plt.tight_layout()
plt.show()

print('[OK] Gamma visualizations complete')

### Practical Example: SPY Straddle Gamma Scalping

**Scenario**: A market maker sells 100 ATM straddles on SPY and delta-hedges daily.

**Position Details:**
- Underlying: SPY @ $450
- Position: Short 100 straddles (ATM, K=$450)
- Time to expiry: 30 days
- Implied vol: 20%
- Delta hedge: Daily rebalancing

**Analysis:**

In [ ]:
# SPY Straddle Gamma Scalping Example

# Position parameters
S0 = 450  # SPY current price
K = 450   # ATM strike
T = 30/252  # 30 days to expiry
r = 0.05
implied_vol = 0.20
n_contracts = 100

# Calculate position Greeks (straddle = call + put)
call_price = black_scholes_call(S0, K, T, r, implied_vol)
put_price = black_scholes_put(S0, K, T, r, implied_vol)
straddle_price = call_price + put_price

# Greeks for straddle (delta = 0 for ATM straddle)
delta_call = delta_bs(S0, K, T, r, implied_vol, 'call')
delta_put = delta_bs(S0, K, T, r, implied_vol, 'put')
delta_straddle = delta_call + delta_put  # Should be ~0 for ATM

gamma_straddle = 2 * gamma_bs(S0, K, T, r, implied_vol)  # Call + Put

# Portfolio level
position_value = n_contracts * straddle_price * 100  # $100 multiplier
position_gamma = n_contracts * gamma_straddle * 100

print('=' * 70)
print('SPY ATM STRADDLE POSITION ANALYSIS')
print('=' * 70)
print(f'\nPosition Details:')
print(f'  Underlying: SPY @ ${S0}')
print(f'  Contracts: {n_contracts} straddles (short)')
print(f'  Strike: ${K} (ATM)')
print(f'  Time to expiry: {int(T*252)} days')
print(f'  Implied Vol: {implied_vol*100:.1f}%')
print(f'\nOption Prices:')
print(f'  Call Price: ${call_price:.2f}')
print(f'  Put Price:  ${put_price:.2f}')
print(f'  Straddle:   ${straddle_price:.2f}')
print(f'\nPosition Greeks:')
print(f'  Delta (portfolio): {delta_straddle:.4f} (near zero for ATM)')
print(f'  Gamma (per straddle): {gamma_straddle:.6f}')
print(f'  Gamma (portfolio): {position_gamma:.2f}')
print(f'\nPosition Value:')
print(f'  Premium collected: ${position_value:,.2f}')

# Simulate gamma P&L over 30 days with different realized vols
print(f'\n{"─" * 70}')
print('GAMMA SCALPING P&L SIMULATION (30 days, daily rebalancing)')
print('─' * 70)

scenarios_results = []
np.random.seed(42)

for realized_vol in [0.15, 0.20, 0.25, 0.30]:
    # Simulate price path
    dt = 1/252
    n_steps = 30
    Z = np.random.randn(n_steps)
    returns = (r - 0.5*realized_vol**2)*dt + realized_vol*np.sqrt(dt)*Z
    S_path = S0 * np.exp(np.cumsum(returns))
    S_path = np.insert(S_path, 0, S0)
    
    # Gamma P&L (short gamma = negative P&L from volatility)
    total_pnl, _ = gamma_scalping_pnl(S_path, position_gamma, 0)
    total_pnl = -total_pnl  # Short position
    
    # Theta P&L (collect time decay)
    theta_pnl = n_steps * 0.5 * implied_vol**2 * S0**2 * position_gamma / 252
    
    net_pnl = total_pnl + theta_pnl
    
    scenarios_results.append({
        'Realized Vol': f'{realized_vol*100:.0f}%',
        'Gamma P&L': total_pnl,
        'Theta P&L': theta_pnl,
        'Net P&L': net_pnl,
        'Profit?': 'Yes' if net_pnl > 0 else 'No'
    })
    
df_scenarios = pd.DataFrame(scenarios_results)
print(df_scenarios.to_string(index=False))

print(f'\n{"─" * 70}')
print('KEY INSIGHTS:')
print('─' * 70)
print('1. SHORT GAMMA: Lose money when realized vol > implied vol')
print('2. THETA BENEFIT: Collect time decay (positive theta for short options)')
print('3. BREAK-EVEN: Profit when realized vol < implied vol')
print(f'4. IMPLIED VOL = {implied_vol*100:.0f}%: Market maker profits if realized < {implied_vol*100:.0f}%')
print('5. RISK: Large moves (gap risk, crashes) can overwhelm theta collection')
print('=' * 70)

## 3. Theta: Time Decay

### Theory: The Cost of Optionality

**Theta (Θ)** measures the rate of change of option value with respect to time:

$$
\Theta = \frac{\partial C}{\partial t}
$$

Note: Often reported as $\frac{\partial C}{\partial T}$ (negative time to expiry), so Theta is usually **negative for long options**.

**Key Properties:**
1. **Negative for long options** - time decay erodes option value
2. **Accelerates near expiry** - most decay in final weeks/days
3. **Related to Gamma** via Black-Scholes PDE: $\Theta \approx -\frac{1}{2}\sigma^2 S^2 \Gamma$
4. **Different for calls and puts** (unlike Gamma)

**Why Theta Matters:**

1. **Option Sellers**: Theta is the "rent" collected for providing insurance
   - Covered calls, cash-secured puts, iron condors
   - Systematic theta collection strategies
2. **Time Decay Patterns**: Non-linear decay (accelerates near expiry)
3. **Weekend Effect**: 3 days of decay on Friday (markets closed Sat/Sun)
4. **Portfolio Management**: Balance theta collection vs gamma risk

**Theta-Gamma-Vega Relationship:**

From Black-Scholes PDE, for a delta-neutral portfolio:
$$
\Theta + \frac{1}{2}\sigma^2 S^2 \Gamma = 0 \quad \text{(simplified)}
$$

This fundamental relationship shows:
- **High gamma → high theta decay**
- Cannot have convexity benefit (gamma) without paying for it (theta)
- Core tradeoff in options trading

### Mathematical Derivation: Black-Scholes Theta

From the Black-Scholes formula:
$$
C = S N(d_1) - K e^{-rT} N(d_2)
$$

Taking the partial derivative with respect to time $T$:
$$
\Theta_{\text{call}} = \frac{\partial C}{\partial T} = -\frac{S \phi(d_1) \sigma}{2\sqrt{T}} - rK e^{-rT} N(d_2)
$$

For a **put option**:
$$
\Theta_{\text{put}} = \frac{\partial P}{\partial T} = -\frac{S \phi(d_1) \sigma}{2\sqrt{T}} + rK e^{-rT} N(-d_2)
$$

where $\phi(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ is the standard normal PDF.

**Common Convention**: Report theta as **daily decay** (per 1 calendar day):
$$
\Theta_{\text{daily}} = \Theta_{\text{annual}} / 365
$$

**Key Differences:**
- **Calls**: More negative theta (pay for upside optionality + interest on strike)
- **Puts**: Less negative theta (interest term partially offsets decay)
- **ATM options**: Highest absolute theta (most time value to decay)
- **Deep ITM/OTM**: Lower theta (less time value)

**Theta Behavior:**
- Theta is **most negative ATM**
- Theta **accelerates as T → 0** (time value decays faster near expiry)
- For European options: Theta can be positive for deep ITM puts (early exercise value)

In [ ]:
# Python Implementation: Theta and Time Decay Simulation

def theta_bs(S: float, K: float, T: float, r: float, sigma: float, option_type: str = 'call') -> float:
    """
    Calculate Black-Scholes Theta (time decay).
    
    Theta measures the rate of change of option value with respect to time.
    Convention: Returns daily theta (divide annual by 365).
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity (years)
    r : float
        Risk-free rate (annualized)
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Daily theta (negative for long options, positive for short)
    
    Notes
    -----
    - Usually negative for long options (time decay)
    - Most negative at ATM
    - Accelerates near expiry
    - Related to gamma: Θ ≈ -0.5·σ²·S²·Γ (delta-neutral)
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Common term (diffusion/gamma part)
    theta_diffusion = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
    
    if option_type == 'call':
        # Call theta: diffusion term - interest on strike
        theta_annual = theta_diffusion - r * K * np.exp(-r * T) * norm.cdf(d2)
    else:
        # Put theta: diffusion term + interest on strike
        theta_annual = theta_diffusion + r * K * np.exp(-r * T) * norm.cdf(-d2)
    
    # Convert to daily theta
    theta_daily = theta_annual / 365
    
    return theta_daily


def rho_bs(S: float, K: float, T: float, r: float, sigma: float, option_type: str = 'call') -> float:
    """
    Calculate Black-Scholes Rho (interest rate sensitivity).
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to maturity (years)
    r : float
        Risk-free rate (annualized)
    sigma : float
        Volatility (annualized)
    option_type : str
        'call' or 'put'
    
    Returns
    -------
    float
        Rho value (change in option value per 1% change in rates)
    
    Notes
    -----
    - Positive for calls (benefit from higher rates)
    - Negative for puts (hurt by higher rates)
    - Scales with time to expiry (longer dated = higher rho)
    """
    if T <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
    
    # Return rho per 1% rate change
    return rho / 100


def theta_decay_simulation(S: float, K: float, T: float, r: float, sigma: float, 
                          option_type: str = 'call', n_days: int = 60) -> pd.DataFrame:
    """
    Simulate theta decay over time for a single option.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Initial time to maturity (years)
    r : float
        Risk-free rate
    sigma : float
        Volatility
    option_type : str
        'call' or 'put'
    n_days : int
        Number of days to simulate
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: days_to_expiry, option_price, daily_theta, cumulative_decay
    """
    results = []
    
    for day in range(n_days, -1, -1):
        T_current = day / 365
        
        if option_type == 'call':
            price = black_scholes_call(S, K, T_current, r, sigma)
        else:
            price = black_scholes_put(S, K, T_current, r, sigma)
        
        if T_current > 0:
            theta = theta_bs(S, K, T_current, r, sigma, option_type)
        else:
            theta = 0
        
        results.append({
            'days_to_expiry': day,
            'time_years': T_current,
            'option_price': price,
            'daily_theta': theta
        })
    
    df = pd.DataFrame(results)
    df['cumulative_decay'] = df['option_price'].iloc[0] - df['option_price']
    
    return df


print('[OK] Theta and Rho functions implemented')

# Test
S, K, T, r, sigma = 100, 100, 0.25, 0.05, 0.25
theta_call = theta_bs(S, K, T, r, sigma, 'call')
theta_put = theta_bs(S, K, T, r, sigma, 'put')
rho_call = rho_bs(S, K, T, r, sigma, 'call')
rho_put = rho_bs(S, K, T, r, sigma, 'put')

print(f'\nTest (S={S}, K={K}, T={T}, r={r}, σ={sigma}):')
print(f'Theta Call: ${theta_call:.4f}/day (time decay)')
print(f'Theta Put:  ${theta_put:.4f}/day')
print(f'Rho Call:   ${rho_call:.4f} per 1% rate change')
print(f'Rho Put:    ${rho_put:.4f} per 1% rate change')

In [ ]:
# Visualization: Theta Behavior (2x2 subplots)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Theta: Time Decay Analysis', fontsize=16, fontweight='bold', y=1.00)

# Base parameters
S0 = 100
K = 100
r = 0.05
sigma = 0.25

# Panel 1: Theta across strikes (different maturities)
ax = axes[0, 0]
spot_range = np.linspace(70, 130, 200)
maturities = [0.08, 0.25, 0.5, 1.0]

for T in maturities:
    thetas_call = [theta_bs(S, K, T, r, sigma, 'call') for S in spot_range]
    ax.plot(spot_range, thetas_call, linewidth=2, label=f'T = {T:.2f}y ({int(T*365)}d)')

ax.axvline(x=K, color='red', linestyle='--', alpha=0.5, label='Strike K=100')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Spot Price ($)', fontsize=11)
ax.set_ylabel('Daily Theta ($)', fontsize=11)
ax.set_title('Theta vs Spot (Call, most negative ATM)', fontsize=12)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Panel 2: Theta decay pattern over time
ax = axes[0, 1]
df_decay = theta_decay_simulation(S0, K, 90/365, r, sigma, 'call', n_days=90)

ax.plot(df_decay['days_to_expiry'], df_decay['option_price'],
        linewidth=2.5, label='Option Price', color=colors[0])
ax2 = ax.twinx()
ax2.plot(df_decay['days_to_expiry'], abs(df_decay['daily_theta']),
         linewidth=2.5, label='|Daily Theta|', color=colors[1], linestyle='--')

ax.set_xlabel('Days to Expiry', fontsize=11)
ax.set_ylabel('Option Price ($)', fontsize=11, color=colors[0])
ax2.set_ylabel('|Daily Theta| ($)', fontsize=11, color=colors[1])
ax.set_title('Theta Acceleration Near Expiry (ATM Call)', fontsize=12)
ax.tick_params(axis='y', labelcolor=colors[0])
ax2.tick_params(axis='y', labelcolor=colors[1])
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# Panel 3: Weekend effect (Friday theta includes 3 days)
ax = axes[1, 0]
days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
T = 30/365

# Regular days: 1 day theta, Friday: 3 days theta
thetas_regular = [theta_bs(S0, K, T, r, sigma, 'call') for _ in range(4)]
theta_friday = 3 * theta_bs(S0, K, T, r, sigma, 'call')
thetas_week = thetas_regular + [theta_friday]

colors_bars = [colors[0]]*4 + [colors[1]]
bars = ax.bar(days_of_week, thetas_week, color=colors_bars, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_ylabel('Daily Theta ($)', fontsize=11)
ax.set_title('Weekend Effect (Friday theta includes 3 days)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:.3f}',
            ha='center', va='bottom' if height < 0 else 'top', fontsize=9)

ax.text(0.02, 0.02, 'Friday includes Sat/Sun decay',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Panel 4: Theta vs moneyness heatmap
ax = axes[1, 1]
spot_range_heat = np.linspace(80, 120, 50)
time_range_heat = np.linspace(0.02, 1.0, 50)

theta_matrix = np.zeros((len(time_range_heat), len(spot_range_heat)))
for i, T in enumerate(time_range_heat):
    for j, S in enumerate(spot_range_heat):
        theta_matrix[i, j] = theta_bs(S, K, T, r, sigma, 'call')

im = ax.contourf(spot_range_heat, time_range_heat*365, theta_matrix, levels=20, cmap='RdYlGn_r')
ax.axvline(x=K, color='white', linestyle='--', linewidth=2, label='ATM (K=100)')
ax.set_xlabel('Spot Price ($)', fontsize=11)
ax.set_ylabel('Days to Expiry', fontsize=11)
ax.set_title('Theta Heatmap (Call, darker = more decay)', fontsize=12)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Daily Theta ($)', fontsize=10)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print('[OK] Theta visualizations complete')

### Practical Example: Covered Call Theta Collection

**Scenario**: Income strategy using covered calls on AAPL stock.

**Position Details:**
- Underlying: Long 1,000 shares AAPL @ $180
- Strategy: Sell 10 OTM calls monthly (K=$190, 30 DTE)
- Objective: Collect theta premium while maintaining upside to $190

In [ ]:
# Covered Call Theta Collection Example

S0, K, T, r, sigma = 180, 190, 30/365, 0.05, 0.30
n_shares, n_contracts = 1000, 10

call_price = black_scholes_call(S0, K, T, r, sigma)
theta_call = theta_bs(S0, K, T, r, sigma, 'call')
premium_collected = n_contracts * call_price * 100

print('=' * 75)
print('COVERED CALL STRATEGY')
print('=' * 75)
print(f'Premium Collected: ${premium_collected:,.2f}')
print(f'Daily Theta: ${-n_contracts * theta_call * 100:.2f}')
print('=' * 75)

## 4. Rho: Interest Rate Sensitivity

**Rho (ρ)** measures sensitivity to interest rate changes. Critical for LEAPS and rate-sensitive products.

$$\rho = \frac{\partial C}{\partial r}$$

- Positive for calls, negative for puts
- Scales with time to expiry

### Black-Scholes Rho Formula

$$\rho_{\text{call}} = KT e^{-rT} N(d_2)$$
$$\rho_{\text{put}} = -KT e^{-rT} N(-d_2)$$

In [ ]:
# Rho is already implemented in rho_bs() function above
# Test it
S, K, T, r, sigma = 100, 100, 2.0, 0.05, 0.25  # 2-year LEAPS
rho_call = rho_bs(S, K, T, r, sigma, 'call')
rho_put = rho_bs(S, K, T, r, sigma, 'put')

print(f'LEAPS Rho (T={T}y):')
print(f'  Call Rho: ${rho_call:.4f} per 1% rate change')
print(f'  Put Rho:  ${rho_put:.4f} per 1% rate change')

In [ ]:
# Visualization: Rho Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Rho: Interest Rate Sensitivity', fontsize=16, fontweight='bold')

# Panel 1: Rho vs Strike
ax = axes[0, 0]
S0, r, sigma = 100, 0.05, 0.25
strikes = np.linspace(70, 130, 100)
for T in [0.5, 1.0, 2.0]:
    rhos = [rho_bs(S0, K, T, r, sigma, 'call') for K in strikes]
    ax.plot(strikes, rhos, linewidth=2, label=f'T={T}y')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Rho (Call)')
ax.set_title('Rho vs Strike (longer dated = higher rho)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Rho vs Time
ax = axes[0, 1]
time_range = np.linspace(0.1, 3.0, 100)
for K in [90, 100, 110]:
    rhos = [rho_bs(S0, K, T, r, sigma, 'call') for T in time_range]
    ax.plot(time_range, rhos, linewidth=2, label=f'K={K}')
ax.set_xlabel('Time to Expiry (years)')
ax.set_ylabel('Rho (Call)')
ax.set_title('Rho vs Time (linear growth)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Rate shock analysis
ax = axes[1, 0]
K, T = 100, 2.0
rate_range = np.linspace(0, 0.10, 50)
call_prices = [black_scholes_call(S0, K, T, r, sigma) for r in rate_range]
put_prices = [black_scholes_put(S0, K, T, r, sigma) for r in rate_range]
ax.plot(rate_range*100, call_prices, linewidth=2, label='Call', color=colors[0])
ax.plot(rate_range*100, put_prices, linewidth=2, label='Put', color=colors[1])
ax.set_xlabel('Interest Rate (%)')
ax.set_ylabel('Option Price ($)')
ax.set_title('Price vs Rates (LEAPS, T=2y)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Rate hike impact
ax = axes[1, 1]
scenarios = ['0% rates', '+1%', '+2%', '+3%', '+4%', '+5%']
rate_changes = [0, 0.01, 0.02, 0.03, 0.04, 0.05]
K, T = 100, 2.0
base_rate = 0.01

call_impacts = []
put_impacts = []
for dr in rate_changes:
    r_new = base_rate + dr
    call_price = black_scholes_call(S0, K, T, r_new, sigma)
    put_price = black_scholes_put(S0, K, T, r_new, sigma)
    call_impacts.append(call_price)
    put_impacts.append(put_price)

x = np.arange(len(scenarios))
width = 0.35
ax.bar(x - width/2, call_impacts, width, label='Call', color=colors[0], alpha=0.7)
ax.bar(x + width/2, put_impacts, width, label='Put', color=colors[1], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, rotation=45)
ax.set_ylabel('Option Price ($)')
ax.set_title('Rate Hike Impact (2Y LEAPS)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('[OK] Rho visualizations complete')

### Practical Example: LEAPS During Fed Rate Hikes

LEAPS position during 2022-2024 rate cycle (0% → 5%)

In [ ]:
# LEAPS Rate Sensitivity Example
S0, K, T, sigma = 150, 150, 2.0, 0.25  # 2-year ATM LEAPS
n_contracts = 50

print('=' * 75)
print('LEAPS POSITION: RATE HIKE IMPACT')
print('=' * 75)

for scenario, r_start, r_end in [('2022-2023', 0.00, 0.03),
                                   ('2023-2024', 0.03, 0.05)]:
    call_start = black_scholes_call(S0, K, T, r_start, sigma)
    call_end = black_scholes_call(S0, K, T, r_end, sigma)
    impact = (call_end - call_start) * n_contracts * 100

    print(f'\n{scenario} (rates: {r_start*100:.0f}% → {r_end*100:.0f}%):')
    print(f'  Call price change: ${call_start:.2f} → ${call_end:.2f}')
    print(f'  Portfolio impact: ${impact:,.0f}')

print('=' * 75)

## 5. Vanna and Volga: Cross-Greeks

### Theory: Spot-Vol Interaction and Vol Convexity

**Vanna** (spot-vol cross-Greek):
$$\text{Vanna} = \frac{\partial^2 C}{\partial S \partial \sigma} = \frac{\partial \Delta}{\partial \sigma} = \frac{\partial \nu}{\partial S}$$

**Volga/Vomma** (vol convexity):
$$\text{Volga} = \frac{\partial^2 C}{\partial \sigma^2} = \frac{\partial \nu}{\partial \sigma}$$

**Why They Matter:**
1. **Volatility Smile Calibration**: Essential for FX options
2. **Vanna-Volga Pricing**: Industry-standard for exotic options
3. **Skew Trading**: Profit from volatility surface moves
4. **Dynamic Hedging**: Understand how vega changes with spot

### Mathematical Derivation

**Vanna:**
$$\text{Vanna} = \frac{\phi(d_1)}{S\sigma^2\sqrt{T}}\left[d_2\right]$$

**Volga:**
$$\text{Volga} = S\sqrt{T}\phi(d_1)\frac{d_1 d_2}{\sigma}$$

**Key Properties:**
- Vanna can be positive or negative (depends on moneyness)
- Volga always positive for long options
- Both peak near ATM

In [ ]:
# Implementation: Vanna and Volga

def vanna_bs(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculate Vanna (spot-vol cross-Greek).

    Vanna = ∂²C/∂S∂σ = ∂Δ/∂σ = ∂ν/∂S

    Returns
    -------
    float
        Vanna value (same for calls and puts)
    """
    if T <= 0:
        return 0.0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    vanna = -norm.pdf(d1) * d2 / sigma
    return vanna


def volga_bs(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculate Volga/Vomma (vol convexity).

    Volga = ∂²C/∂σ² = ∂ν/∂σ

    Returns
    -------
    float
        Volga value (same for calls and puts, always positive)
    """
    if T <= 0:
        return 0.0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    volga = S * np.sqrt(T) * norm.pdf(d1) * d1 * d2 / sigma
    return volga


def vega_bs(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Calculate Vega (vol sensitivity)."""
    if T <= 0:
        return 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T) / 100  # Per 1% vol change


print('[OK] Vanna and Volga implemented')

# Test
S, K, T, r, sigma = 100, 100, 0.5, 0.05, 0.25
vanna = vanna_bs(S, K, T, r, sigma)
volga = volga_bs(S, K, T, r, sigma)
vega = vega_bs(S, K, T, r, sigma)

print(f'\nTest (S={S}, K={K}, T={T}):')
print(f'Vega:  {vega:.6f}')
print(f'Vanna: {vanna:.6f}')
print(f'Volga: {volga:.6f}')

In [ ]:
# Visualization: Vanna and Volga
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Vanna and Volga: Cross-Greeks', fontsize=16, fontweight='bold')

S0, r, sigma, T = 100, 0.05, 0.25, 0.5

# Panel 1: Vanna across strikes
ax = axes[0, 0]
strikes = np.linspace(70, 130, 100)
vannas = [vanna_bs(S0, K, T, r, sigma) for K in strikes]
ax.plot(strikes, vannas, linewidth=2, color=colors[0])
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.axvline(x=S0, color='red', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Vanna')
ax.set_title('Vanna vs Strike (changes sign at ATM)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Volga across strikes
ax = axes[0, 1]
volgas = [volga_bs(S0, K, T, r, sigma) for K in strikes]
ax.plot(strikes, volgas, linewidth=2, color=colors[1])
ax.axvline(x=S0, color='red', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Volga')
ax.set_title('Volga vs Strike (peaks ATM, always positive)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Vanna surface (S vs σ)
ax = axes[1, 0]
spot_range = np.linspace(80, 120, 50)
vol_range = np.linspace(0.10, 0.50, 50)
K, T = 100, 0.5

vanna_matrix = np.zeros((len(vol_range), len(spot_range)))
for i, vol in enumerate(vol_range):
    for j, S in enumerate(spot_range):
        vanna_matrix[i, j] = vanna_bs(S, K, T, r, vol)

im = ax.contourf(spot_range, vol_range*100, vanna_matrix, levels=20, cmap='RdBu_r')
ax.axvline(x=K, color='white', linestyle='--', linewidth=2)
ax.axhline(y=sigma*100, color='white', linestyle='--', linewidth=2)
ax.set_xlabel('Spot Price')
ax.set_ylabel('Volatility (%)')
ax.set_title('Vanna Surface (Spot vs Vol)')
plt.colorbar(im, ax=ax, label='Vanna')

# Panel 4: Volga surface
ax = axes[1, 1]
volga_matrix = np.zeros((len(vol_range), len(spot_range)))
for i, vol in enumerate(vol_range):
    for j, S in enumerate(spot_range):
        volga_matrix[i, j] = volga_bs(S, K, T, r, vol)

im = ax.contourf(spot_range, vol_range*100, volga_matrix, levels=20, cmap='YlOrRd')
ax.axvline(x=K, color='white', linestyle='--', linewidth=2)
ax.set_xlabel('Spot Price')
ax.set_ylabel('Volatility (%)')
ax.set_title('Volga Surface (always positive)')
plt.colorbar(im, ax=ax, label='Volga')

plt.tight_layout()
plt.show()
print('[OK] Vanna/Volga visualizations complete')

### Practical Example: Vanna-Volga Pricing for FX Options

Industry-standard method for pricing OTM EUR/USD options.

In [ ]:
# Vanna-Volga Pricing Example
S0, K_atm, T, r = 1.10, 1.10, 0.25, 0.05  # EUR/USD
K_target = 1.15  # OTM call

# Market volatilities (smile)
sigma_atm = 0.10
sigma_25delta = 0.12  # 25-delta risk reversal
sigma_10delta = 0.15  # 10-delta butterfly

# Black-Scholes price (flat vol)
price_bs = black_scholes_call(S0, K_target, T, r, sigma_atm)

# Vanna-Volga correction (simplified)
vanna = vanna_bs(S0, K_target, T, r, sigma_atm)
volga = volga_bs(S0, K_target, T, r, sigma_atm)
vol_correction = volga * (sigma_25delta - sigma_atm)**2 / 2

price_vv = price_bs + vol_correction

print('=' * 75)
print('VANNA-VOLGA PRICING: EUR/USD OPTIONS')
print('=' * 75)
print(f'Spot: {S0:.4f}')
print(f'Strike: {K_target:.4f} (OTM)')
print(f'\nBlack-Scholes (flat vol): ${price_bs:.6f}')
print(f'Vol correction: ${vol_correction:.6f}')
print(f'Vanna-Volga price: ${price_vv:.6f}')
print(f'\nImpact: {(price_vv/price_bs-1)*100:.2f}% adjustment for smile')
print('=' * 75)

## 6. Charm and Veta: Time-Decay Cross-Greeks

### Theory: How Greeks Evolve Over Time

**Charm** (delta decay):
$$\text{Charm} = \frac{\partial^2 C}{\partial S \partial t} = \frac{\partial \Delta}{\partial t} = \frac{\partial \Theta}{\partial S}$$

**Veta/DvegaDtime** (vega decay):
$$\text{Veta} = \frac{\partial^2 C}{\partial \sigma \partial t} = \frac{\partial \nu}{\partial t} = \frac{\partial \Theta}{\partial \sigma}$$

**Why They Matter:**
1. **Dynamic Hedging**: Understand how delta changes over time (not just with spot)
2. **Rebalancing Frequency**: Charm tells you how quickly delta hedge degrades
3. **Vega Management**: Veta shows how vega exposure changes approaching expiry
4. **Calendar Effects**: Greeks evolution over weekends, holidays

**Intuition:**
- **Charm**: As time passes, ATM deltas move toward 0.5, OTM → 0, ITM → 1
- **Veta**: Vega decreases as expiry approaches (faster for ATM)

### Mathematical Derivation

**Charm** (Call):
$$\text{Charm}_{	ext{call}} = -\phi(d_1) \frac{2rT - d_2 \sigma \sqrt{T}}{2T\sigma\sqrt{T}} - r e^{-rT} N(d_2)$$

Simplified approximation:
$$\text{Charm} \approx -\frac{\phi(d_1)}{2T\sigma\sqrt{T}}[2(r-q)T + d_2\sigma\sqrt{T}]$$

**Veta**:
$$\text{Veta} = -S \phi(d_1) \sqrt{T} \frac{d_1 d_2}{\sigma}$$

**Key Properties:**
- Charm changes sign around ATM (ITM calls have negative charm)
- Veta is negative (vega decays over time)
- Both accelerate near expiry

In [ ]:
# Implementation: Charm and Veta

def charm_bs(S: float, K: float, T: float, r: float, sigma: float, option_type: str = 'call') -> float:
    """
    Calculate Charm (delta decay).

    Charm = ∂²C/∂S∂t = ∂Δ/∂t

    Returns
    -------
    float
        Charm value (annual, divide by 365 for daily)
    """
    if T <= 0:
        return 0.0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    term1 = -norm.pdf(d1) * (2*r*T - d2*sigma*np.sqrt(T)) / (2*T*sigma*np.sqrt(T))

    if option_type == 'call':
        term2 = r * np.exp(-r * T) * norm.cdf(d2)
        charm = term1 - term2
    else:
        term2 = r * np.exp(-r * T) * norm.cdf(-d2)
        charm = term1 + term2

    return charm / 365  # Daily charm


def veta_bs(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """
    Calculate Veta (vega decay).

    Veta = ∂²C/∂σ∂t = ∂ν/∂t

    Returns
    -------
    float
        Veta value (same for calls and puts)
    """
    if T <= 0:
        return 0.0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    veta = -S * norm.pdf(d1) * np.sqrt(T) * d1 * d2 / sigma

    return veta / 365  # Daily veta


print('[OK] Charm and Veta implemented')

# Test
S, K, T, r, sigma = 100, 100, 0.25, 0.05, 0.25
charm_call = charm_bs(S, K, T, r, sigma, 'call')
charm_put = charm_bs(S, K, T, r, sigma, 'put')
veta = veta_bs(S, K, T, r, sigma)

print(f'\nTest (S={S}, K={K}, T={T}):')
print(f'Charm Call: {charm_call:.6f} (daily delta decay)')
print(f'Charm Put:  {charm_put:.6f}')
print(f'Veta:       {veta:.6f} (daily vega decay)')

In [ ]:
# Visualization: Charm and Veta
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Charm and Veta: Time-Decay Cross-Greeks', fontsize=16, fontweight='bold')

S0, r, sigma, T = 100, 0.05, 0.25, 0.25

# Panel 1: Charm across strikes
ax = axes[0, 0]
strikes = np.linspace(70, 130, 100)
charms_call = [charm_bs(S0, K, T, r, sigma, 'call') for K in strikes]
charms_put = [charm_bs(S0, K, T, r, sigma, 'put') for K in strikes]
ax.plot(strikes, charms_call, linewidth=2, label='Call', color=colors[0])
ax.plot(strikes, charms_put, linewidth=2, label='Put', color=colors[1])
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.axvline(x=S0, color='red', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Charm (daily)')
ax.set_title('Charm vs Strike (delta decay rate)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Veta across strikes
ax = axes[0, 1]
vetas = [veta_bs(S0, K, T, r, sigma) for K in strikes]
ax.plot(strikes, vetas, linewidth=2, color=colors[2])
ax.axvline(x=S0, color='red', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Veta (daily)')
ax.set_title('Veta vs Strike (vega decay, peaks ATM)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Greeks evolution over time (ATM option)
ax = axes[1, 0]
days_range = np.arange(0, 91)
K = S0

deltas, vegas, thetas = [], [], []
for day in days_range:
    T_curr = (90 - day) / 365
    if T_curr > 0:
        deltas.append(delta_bs(S0, K, T_curr, r, sigma, 'call'))
        vegas.append(vega_bs(S0, K, T_curr, r, sigma))
        thetas.append(abs(theta_bs(S0, K, T_curr, r, sigma, 'call')))
    else:
        deltas.append(1.0 if S0 > K else 0.0)
        vegas.append(0)
        thetas.append(0)

ax.plot(days_range, deltas, linewidth=2, label='Delta', color=colors[0])
ax2 = ax.twinx()
ax2.plot(days_range, vegas, linewidth=2, label='Vega', color=colors[1], linestyle='--')
ax.set_xlabel('Days Elapsed')
ax.set_ylabel('Delta', color=colors[0])
ax2.set_ylabel('Vega', color=colors[1])
ax.set_title('Greeks Evolution Over Time (ATM Call)')
ax.tick_params(axis='y', labelcolor=colors[0])
ax2.tick_params(axis='y', labelcolor=colors[1])
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Panel 4: Charm surface
ax = axes[1, 1]
spot_range = np.linspace(80, 120, 50)
time_range_days = np.linspace(1, 90, 50)

charm_matrix = np.zeros((len(time_range_days), len(spot_range)))
for i, days in enumerate(time_range_days):
    T_curr = days / 365
    for j, S in enumerate(spot_range):
        charm_matrix[i, j] = charm_bs(S, S0, T_curr, r, sigma, 'call')

im = ax.contourf(spot_range, time_range_days, charm_matrix, levels=20, cmap='RdBu_r')
ax.axvline(x=S0, color='white', linestyle='--', linewidth=2)
ax.set_xlabel('Spot Price')
ax.set_ylabel('Days to Expiry')
ax.set_title('Charm Surface (delta decay rate)')
plt.colorbar(im, ax=ax, label='Charm')

plt.tight_layout()
plt.show()
print('[OK] Charm/Veta visualizations complete')

### Practical Example: Dynamic Delta Hedging with Charm

Adjust delta hedge accounting for charm (delta decay).

In [ ]:
# Dynamic Delta Hedging with Charm Example

S0, K, T, r, sigma = 100, 100, 30/365, 0.05, 0.25
n_contracts = 100

# Initial Greeks
delta_0 = delta_bs(S0, K, T, r, sigma, 'call')
charm_0 = charm_bs(S0, K, T, r, sigma, 'call')

print('=' * 75)
print('DYNAMIC DELTA HEDGING WITH CHARM ADJUSTMENT')
print('=' * 75)
print(f'\nInitial Position ({n_contracts} ATM calls):')
print(f'  Spot: ${S0}')
print(f'  Delta: {delta_0:.4f} per contract')
print(f'  Charm: {charm_0:.6f} per contract per day')
print(f'\nPortfolio:')
print(f'  Net Delta: {n_contracts * delta_0:.0f} shares')
print(f'  Daily Delta Decay: {n_contracts * charm_0:.2f} shares/day')

# Simulate 5 days
print(f'\n{"=" * 75}')
print('DELTA EVOLUTION (without spot moves)')
print('=' * 75)

for day in range(6):
    T_curr = max(0, (30 - day) / 365)
    if T_curr > 0:
        delta = delta_bs(S0, K, T_curr, r, sigma, 'call')
        charm = charm_bs(S0, K, T_curr, r, sigma, 'call')
    else:
        delta = 1.0 if S0 > K else 0.0
        charm = 0

    print(f'Day {day}: Delta = {delta:.4f}, Charm = {charm:.6f}')

print(f'\n{"=" * 75}')
print('KEY INSIGHT: Even without spot moves, delta changes due to time decay')
print('Charm quantifies this effect, critical for dynamic hedging strategies')
print('=' * 75)

## 7. Greeks Interactions and P&L Attribution

### Theory: Taylor Series Decomposition

The complete option price change can be expressed using Taylor series expansion:

$$
\Delta C \approx \Delta \cdot \Delta S + \frac{1}{2}\Gamma \cdot (\Delta S)^2 + \nu \cdot \Delta \sigma + \Theta \cdot \Delta t + \rho \cdot \Delta r
$$

$$
+ \text{Vanna} \cdot \Delta S \cdot \Delta \sigma + \frac{1}{2}\text{Volga} \cdot (\Delta \sigma)^2 + \text{Charm} \cdot \Delta S \cdot \Delta t + \ldots
$$

**Complete Greeks Summary:**

| Greek | Definition | Interpretation | Sign |
|-------|-----------|----------------|------|
| **First-Order** |
| Delta (Δ) | ∂C/∂S | Spot sensitivity | Call: +, Put: - |
| Vega (ν) | ∂C/∂σ | Vol sensitivity | Always + |
| Theta (Θ) | ∂C/∂t | Time decay | Usually - |
| Rho (ρ) | ∂C/∂r | Rate sensitivity | Call: +, Put: - |
| **Second-Order** |
| Gamma (Γ) | ∂²C/∂S² | Delta convexity | Always + |
| Volga | ∂²C/∂σ² | Vega convexity | Always + |
| **Cross-Greeks** |
| Vanna | ∂²C/∂S∂σ | Spot-vol interaction | +/- |
| Charm | ∂²C/∂S∂t | Delta decay | +/- |
| Veta | ∂²C/∂σ∂t | Vega decay | Usually - |

### Complete Black-Scholes Greeks Formulas

**First-Order:**
- $\Delta_{\text{call}} = N(d_1)$, $\Delta_{\text{put}} = N(d_1) - 1$
- $\nu = S\phi(d_1)\sqrt{T}$
- $\Theta_{\text{call}} = -\frac{S\phi(d_1)\sigma}{2\sqrt{T}} - rKe^{-rT}N(d_2)$
- $\rho_{\text{call}} = KTe^{-rT}N(d_2)$

**Second-Order:**
- $\Gamma = \frac{\phi(d_1)}{S\sigma\sqrt{T}}$
- $\text{Volga} = S\sqrt{T}\phi(d_1)\frac{d_1 d_2}{\sigma}$

**Cross-Greeks:**
- $\text{Vanna} = -\phi(d_1)\frac{d_2}{\sigma}$
- $\text{Charm} = -\phi(d_1)\frac{2rT - d_2\sigma\sqrt{T}}{2T\sigma\sqrt{T}}$
- $\text{Veta} = -S\phi(d_1)\sqrt{T}\frac{d_1 d_2}{\sigma}$

where $d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$

In [ ]:
# Complete Greeks Package

def all_greeks_bs(S: float, K: float, T: float, r: float, sigma: float, option_type: str = 'call') -> Dict[str, float]:
    """
    Calculate all Black-Scholes Greeks at once.

    Returns
    -------
    dict
        Dictionary with all Greeks
    """
    # Price
    if option_type == 'call':
        price = black_scholes_call(S, K, T, r, sigma)
    else:
        price = black_scholes_put(S, K, T, r, sigma)

    # First-order Greeks
    delta = delta_bs(S, K, T, r, sigma, option_type)
    vega = vega_bs(S, K, T, r, sigma)
    theta = theta_bs(S, K, T, r, sigma, option_type)
    rho = rho_bs(S, K, T, r, sigma, option_type)

    # Second-order Greeks
    gamma = gamma_bs(S, K, T, r, sigma)
    volga = volga_bs(S, K, T, r, sigma)

    # Cross-Greeks
    vanna = vanna_bs(S, K, T, r, sigma)
    charm = charm_bs(S, K, T, r, sigma, option_type)
    veta = veta_bs(S, K, T, r, sigma)

    return {
        'price': price,
        'delta': delta,
        'gamma': gamma,
        'vega': vega,
        'theta': theta,
        'rho': rho,
        'vanna': vanna,
        'volga': volga,
        'charm': charm,
        'veta': veta
    }


def greeks_pnl_attribution(greeks: Dict[str, float], dS: float, dsigma: float, dt: float, dr: float = 0) -> Dict[str, float]:
    """
    Decompose P&L using Taylor series expansion.

    Parameters
    ----------
    greeks : dict
        Greeks dictionary from all_greeks_bs()
    dS : float
        Spot price change
    dsigma : float
        Volatility change (absolute, e.g., 0.02 for 2%)
    dt : float
        Time change (days)
    dr : float
        Rate change (absolute)

    Returns
    -------
    dict
        P&L attribution by Greek
    """
    dt_years = dt / 365

    pnl = {
        'delta': greeks['delta'] * dS,
        'gamma': 0.5 * greeks['gamma'] * dS**2,
        'vega': greeks['vega'] * dsigma * 100,  # Vega per 1% vol
        'theta': greeks['theta'] * dt,
        'rho': greeks['rho'] * dr * 100,  # Rho per 1% rate
        'vanna': greeks['vanna'] * dS * dsigma,
        'volga': 0.5 * greeks['volga'] * dsigma**2,
        'charm': greeks['charm'] * dS * dt,
        'veta': greeks['veta'] * dsigma * dt
    }

    pnl['total'] = sum(pnl.values())

    return pnl


print('[OK] Complete Greeks package implemented')

# Test
S, K, T, r, sigma = 100, 100, 0.25, 0.05, 0.25
greeks = all_greeks_bs(S, K, T, r, sigma, 'call')

print('\nComplete Greeks (ATM Call):')
for name, value in greeks.items():
    print(f'  {name:10s}: {value:10.6f}')

In [ ]:
# Visualization: Greeks Interactions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Greeks Interactions and P&L Attribution', fontsize=16, fontweight='bold')

S0, K, T, r, sigma = 100, 100, 0.25, 0.05, 0.25

# Panel 1: Greeks correlation matrix
ax = axes[0, 0]
strikes = np.linspace(80, 120, 50)
greeks_matrix = []

for strike in strikes:
    g = all_greeks_bs(S0, strike, T, r, sigma, 'call')
    greeks_matrix.append([g['delta'], g['gamma'], g['vega'], g['theta']])

greeks_df = pd.DataFrame(greeks_matrix, columns=['Delta', 'Gamma', 'Vega', 'Theta'])
corr = greeks_df.corr()

im = ax.imshow(corr, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        text = ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                      ha='center', va='center', color='black', fontsize=10)

ax.set_title('Greeks Correlation Matrix')
plt.colorbar(im, ax=ax)

# Panel 2: P&L attribution pie chart
ax = axes[0, 1]
greeks = all_greeks_bs(S0, K, T, r, sigma, 'call')
pnl = greeks_pnl_attribution(greeks, dS=2, dsigma=0.02, dt=1)

pnl_positive = {k: v for k, v in pnl.items() if v > 0 and k != 'total'}
pnl_negative = {k: abs(v) for k, v in pnl.items() if v < 0 and k != 'total'}

if pnl_positive:
    wedges, texts, autotexts = ax.pie(pnl_positive.values(), labels=pnl_positive.keys(),
                                        autopct='%1.1f%%', startangle=90, colors=colors)
    ax.set_title(f'P&L Attribution (dS=+$2, dσ=+2%, 1 day)\nTotal: ${pnl["total"]:.2f}')

# Panel 3: Greeks evolution heatmap
ax = axes[1, 0]
days_range = np.arange(1, 91)
strikes_range = np.linspace(90, 110, 30)

gamma_matrix = np.zeros((len(days_range), len(strikes_range)))
for i, days in enumerate(days_range):
    T_curr = days / 365
    for j, strike in enumerate(strikes_range):
        gamma_matrix[i, j] = gamma_bs(S0, strike, T_curr, r, sigma)

im = ax.contourf(strikes_range, days_range, gamma_matrix, levels=20, cmap='YlOrRd')
ax.set_xlabel('Strike Price')
ax.set_ylabel('Days to Expiry')
ax.set_title('Gamma Heatmap (Time vs Strike)')
plt.colorbar(im, ax=ax, label='Gamma')

# Panel 4: Multi-scenario P&L attribution
ax = axes[1, 1]
scenarios = [
    ('Up+VolUp', 3, 0.03, colors[0]),
    ('Up+VolDown', 3, -0.03, colors[1]),
    ('Down+VolUp', -3, 0.03, colors[2]),
    ('Down+VolDown', -3, -0.03, colors[3])
]

pnl_components = ['delta', 'gamma', 'vega', 'theta']
x = np.arange(len(scenarios))
width = 0.2

for i, component in enumerate(pnl_components):
    values = []
    for _, dS, dsig, _ in scenarios:
        pnl = greeks_pnl_attribution(greeks, dS, dsig, 1)
        values.append(pnl[component])

    ax.bar(x + i*width, values, width, label=component.capitalize())

ax.set_xlabel('Scenario')
ax.set_ylabel('P&L ($)')
ax.set_title('P&L Attribution by Scenario')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([s[0] for s in scenarios], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)

plt.tight_layout()
plt.show()
print('[OK] Greeks interaction visualizations complete')

### Practical Example: Explain Daily P&L

Decompose a portfolio's 1-day P&L using complete Greeks.

In [ ]:
# Daily P&L Attribution Example

# Yesterday's position
S_yesterday = 100
sigma_yesterday = 0.25
K, T_yesterday, r = 100, 30/365, 0.05

# Today's market
S_today = 102
sigma_today = 0.27
T_today = 29/365

# Calculate Greeks yesterday
greeks_yesterday = all_greeks_bs(S_yesterday, K, T_yesterday, r, sigma_yesterday, 'call')

# Calculate actual prices
price_yesterday = greeks_yesterday['price']
price_today = black_scholes_call(S_today, K, T_today, r, sigma_today)
actual_pnl = price_today - price_yesterday

# Market moves
dS = S_today - S_yesterday
dsigma = sigma_today - sigma_yesterday
dt = 1  # day

# P&L attribution
pnl = greeks_pnl_attribution(greeks_yesterday, dS, dsigma, dt)

print('=' * 75)
print('DAILY P&L ATTRIBUTION')
print('=' * 75)
print(f'\nMarket Moves:')
print(f'  Spot: ${S_yesterday} -> ${S_today} (${dS:+.2f})')
print(f'  Vol:  {sigma_yesterday*100:.1f}% -> {sigma_today*100:.1f}% ({dsigma*100:+.1f}%)')
print(f'  Time: {T_yesterday*365:.0f}d -> {T_today*365:.0f}d (-1 day)')
print(f'\nActual P&L: ${actual_pnl:.4f}')
print(f'\nGreeks-Based Attribution:')
print(f'  Delta P&L:  ${pnl["delta"]:8.4f} ({abs(pnl["delta"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  Gamma P&L:  ${pnl["gamma"]:8.4f} ({abs(pnl["gamma"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  Vega P&L:   ${pnl["vega"]:8.4f} ({abs(pnl["vega"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  Theta P&L:  ${pnl["theta"]:8.4f} ({abs(pnl["theta"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  Vanna P&L:  ${pnl["vanna"]:8.4f} ({abs(pnl["vanna"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  Volga P&L:  ${pnl["volga"]:8.4f} ({abs(pnl["volga"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  Charm P&L:  ${pnl["charm"]:8.4f} ({abs(pnl["charm"])/abs(actual_pnl)*100:5.1f}%)')
print(f'  {"─"*35}')
print(f'  Total:      ${pnl["total"]:8.4f}')
print(f'\nExplained:   {pnl["total"]/actual_pnl*100:.1f}% of actual P&L')
print(f'Residual:    ${actual_pnl - pnl["total"]:.4f} (higher-order terms)')
print('=' * 75)

## 8. Comprehensive Case Study: Options Market Maker Portfolio

### Scenario

You are a market maker managing a complex SPY options portfolio:

**Portfolio Composition:**
1. Long 100 ATM straddles (K=$450, 30 DTE)
2. Short 200 OTM strangles (K=$430/$470, 30 DTE)
3. Flat delta via stock hedge

**Market Scenario:**
- Current SPY: $450
- Current IV: 20%
- Interest rate: 5%

**Tasks:**
1. Calculate complete Greeks profile
2. Simulate market shocks (spot ±5%, vol ±20%, 1 day)
3. P&L attribution using Taylor series
4. Risk report with Greeks heatmap
5. Dynamic hedging recommendations

In [ ]:
# Comprehensive Case Study Implementation

class OptionsPortfolioManager:
    def __init__(self, S, r, sigma, T):
        self.S = S
        self.r = r
        self.sigma = sigma
        self.T = T
        self.positions = []

    def add_position(self, option_type, K, quantity, position_type='long'):
        """Add option position to portfolio."""
        multiplier = 1 if position_type == 'long' else -1
        self.positions.append({
            'option_type': option_type,
            'K': K,
            'quantity': quantity * multiplier
        })

    def calculate_portfolio_greeks(self):
        """Calculate aggregate Greeks for entire portfolio."""
        portfolio_greeks = {
            'price': 0, 'delta': 0, 'gamma': 0, 'vega': 0,
            'theta': 0, 'rho': 0, 'vanna': 0, 'volga': 0,
            'charm': 0, 'veta': 0
        }

        for pos in self.positions:
            greeks = all_greeks_bs(self.S, pos['K'], self.T, self.r,
                                   self.sigma, pos['option_type'])

            for key in portfolio_greeks:
                portfolio_greeks[key] += greeks[key] * pos['quantity'] * 100

        return portfolio_greeks

    def simulate_scenario(self, dS_pct, dsigma_pct, dt_days):
        """Simulate P&L under market scenario."""
        greeks = self.calculate_portfolio_greeks()

        dS = self.S * dS_pct
        dsigma = self.sigma * dsigma_pct

        pnl = greeks_pnl_attribution(
            {k: v/100 for k, v in greeks.items()},  # Per contract
            dS, dsigma, dt_days
        )

        # Scale to portfolio
        return {k: v * sum(abs(p['quantity']) for p in self.positions)
                for k, v in pnl.items()}

# Initialize portfolio
S0, r, sigma, T = 450, 0.05, 0.20, 30/365
portfolio = OptionsPortfolioManager(S0, r, sigma, T)

# Add positions
# Long 100 ATM straddles
portfolio.add_position('call', 450, 100, 'long')
portfolio.add_position('put', 450, 100, 'long')

# Short 200 OTM strangles
portfolio.add_position('call', 470, 200, 'short')
portfolio.add_position('put', 430, 200, 'short')

# Calculate Greeks
greeks = portfolio.calculate_portfolio_greeks()

print('=' * 75)
print('OPTIONS MARKET MAKER PORTFOLIO ANALYSIS')
print('=' * 75)
print(f'\nPortfolio Composition:')
print(f'  Long 100 ATM straddles (K=$450)')
print(f'  Short 200 OTM strangles (K=$430/$470)')
print(f'\nPortfolio Greeks:')
print(f'  Delta:  {greeks["delta"]:10.2f} shares')
print(f'  Gamma:  {greeks["gamma"]:10.4f}')
print(f'  Vega:   {greeks["vega"]:10.2f} (per 1% vol)')
print(f'  Theta:  {greeks["theta"]:10.2f} per day')
print(f'  Vanna:  {greeks["vanna"]:10.4f}')
print(f'  Volga:  {greeks["volga"]:10.4f}')

# Stress testing
print(f'\n{"=" * 75}')
print('STRESS TEST: P&L UNDER VARIOUS SCENARIOS')
print('=' * 75)

scenarios = [
    ('Down 5%, Vol -20%', -0.05, -0.20),
    ('Down 5%, Vol +20%', -0.05, 0.20),
    ('Flat, Vol +20%', 0, 0.20),
    ('Up 5%, Vol -20%', 0.05, -0.20),
    ('Up 5%, Vol +20%', 0.05, 0.20),
]

results = []
for name, dS_pct, dvol_pct in scenarios:
    pnl = portfolio.simulate_scenario(dS_pct, dvol_pct, 1)
    results.append({
        'Scenario': name,
        'Total P&L': f'${pnl["total"]:,.0f}',
        'Delta': f'${pnl["delta"]:,.0f}',
        'Gamma': f'${pnl["gamma"]:,.0f}',
        'Vega': f'${pnl["vega"]:,.0f}',
        'Theta': f'${pnl["theta"]:,.0f}'
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print(f'\n{"=" * 75}')
print('RISK MANAGEMENT RECOMMENDATIONS:')
print('=' * 75)
print(f'1. Delta: {greeks["delta"]:.0f} shares -> Hedge with {-greeks["delta"]:.0f} SPY shares')
print(f'2. Gamma: {greeks["gamma"]:.2f} -> {"Positive" if greeks["gamma"] > 0 else "Negative"} convexity')
print(f'3. Vega: {greeks["vega"]:.0f} -> Exposed to vol {"increase" if greeks["vega"] > 0 else "decrease"}')
print(f'4. Theta: {greeks["theta"]:.0f}/day -> {"Collecting" if greeks["theta"] > 0 else "Paying"} time decay')
print('5. Monitor vol smile changes (Vanna/Volga exposure)')
print('=' * 75)

## Practice Exercises\n\nTest your understanding with these exercises:\n\n### Exercise 1\nDescription of exercise 1...\n\n### Exercise 2\nDescription of exercise 2...\n\n### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



## Summary and Key Takeaways

### Complete Greeks Reference

| Greek | Formula | When It Matters | Typical Trading Strategy |
|-------|---------|----------------|-------------------------|
| **Gamma** | φ(d₁)/(Sσ√T) | Short-dated ATM, vol trading | Gamma scalping, straddles |
| **Theta** | -Sφ(d₁)σ/(2√T) - rKe⁻ʳᵀN(d₂) | All options (time decay) | Covered calls, iron condors |
| **Rho** | KTe⁻ʳᵀN(d₂) | LEAPS, rate changes | Long-term hedges |
| **Vanna** | -φ(d₁)d₂/σ | Vol smile trading, FX | Vanna-volga pricing |
| **Volga** | Sφ(d₁)√T·d₁d₂/σ | Vol-of-vol trading | Volatility swaps |
| **Charm** | -φ(d₁)·(...) | Dynamic hedging | Delta decay management |
| **Veta** | -Sφ(d₁)√T·d₁d₂/σ | Vega management | Time-varying vega hedge |

### When Each Greek Matters Most

1. **Gamma**: Market makers, short-dated options, gamma scalping strategies
2. **Theta**: Option sellers, income strategies, theta decay harvesting
3. **Rho**: LEAPS traders, rising rate environments, long-dated positions
4. **Vanna**: FX options, vol smile trading, skew exposure management
5. **Volga**: Vol-of-vol trading, exotic options, smile calibration
6. **Charm**: Dynamic hedging, understanding delta decay without spot moves
7. **Veta**: Long vega positions, understanding how vega changes with time

### Cross-Greeks: Why They Matter

Cross-Greeks capture **interaction effects** between market variables:
- **Vanna**: Delta changes with volatility (spot-vol correlation)
- **Charm**: Delta decays over time (without spot moves)
- **Veta**: Vega decays over time

Essential for:
- Accurate P&L attribution
- Volatility smile dynamics
- Second-order hedging
- Exotic options pricing

### Trading Strategies by Greek

1. **Long Gamma**: Buy straddles/strangles, profit from volatility
2. **Short Gamma**: Sell options, profit from range-bound markets
3. **Theta Collection**: Covered calls, cash-secured puts, iron condors
4. **Vanna Trading**: Risk reversals, exploiting spot-vol correlation
5. **Volga Trading**: Butterfly spreads, volatility smile arbitrage

### Academic References

1. **Taleb, N.** (1997). *Dynamic Hedging: Managing Vanilla and Exotic Options*. Wiley.
   - Comprehensive Greeks treatment, practical hedging

2. **Haug, E.G.** (2007). *The Complete Guide to Option Pricing Formulas*. McGraw-Hill.
   - All Greeks formulas, numerical methods

3. **Gatheral, J.** (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley.
   - Volatility smile, Vanna-Volga pricing

4. **Carr, P. & Madan, D.** (1999). "Option Valuation Using the Fast Fourier Transform." *Journal of Computational Finance*.
   - Model-free implied volatility

5. **Wystup, U.** (2006). *FX Options and Structured Products*. Wiley.
   - Vanna-Volga methodology, market conventions

6. **Hull, J.C.** (2022). *Options, Futures, and Other Derivatives* (11th ed.). Pearson.
   - Standard reference, Greeks intuition

### Next Steps

1. **Practice**: Implement Greeks calculators for your favorite underlyings
2. **Backtest**: Test gamma scalping strategies on historical data
3. **Monitor**: Track real-world Greeks evolution on live positions
4. **Advanced Topics**:
   - Implied volatility surfaces
   - Model-free Greeks (variance swaps)
   - Machine learning for Greeks prediction
   - High-order Greeks (Speed, Color, Zomma)

---

**Congratulations!** You now have a complete understanding of higher-order Greeks and cross-Greeks, from mathematical foundations to practical trading applications.